In [34]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random()<0.05:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.01, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'hour': random.randint(0, 5),
            'timestamp': datetime.now().isoformat(),
        }
    else:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'hour': random.randint(7, 23),
            'timestamp': datetime.now().isoformat(),
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [35]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    if message.value['amount'] > 1000:
        print(f'ALERT {message.value}')
    

Overwriting consumer_filter.py


In [36]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(|
    'transactions',
    bootstrap_server='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    if message.value['amount'] > 3000:
        message.value['risk_level'] = 'HIGH'
    elif message.value['amount'] > 1000:
        message.value['risk_level'] = 'MEDIUM'
    else:
        message.value['risk_level'] = 'LOW'
    print(message.value)

Overwriting consumer_enrich.py


In [37]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    store = message.value['store']
    amount = message.value['amount']
    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1
    if msg_count % 10 == 0:
        for store in store_counts:
            print(f'Message counter: {msg_count} || Store name: {store:<12} || Store counts: {store_counts[store]:<20} || Total amount: {total_amount[store]}')

Overwriting consumer_count.py


In [27]:
# %%file score_transaction.py
from datetime import datetime

def score_transaction(tx):
    score = 0
    rules = []

    dt = datetime.fromisoformat(tx['timestamp'])
    hour = dt.hour
    if hour < 6:
        score += 2
        rules.append('R3')

    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')

    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')
    
    return score, rules

# Test
test_tx = {'tx_id': 'TX999', 'amount': 4500.0, 'category': 'elektronika',
           'timestamp': '2026-04-01T03:15:00'}
print(score_transaction(test_tx))  # powinno dać score >= 5

(7, ['R3', 'R2', 'R1'])


In [40]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json
from datetime import datetime

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

def score_transaction(tx):
    score = 0
    rules = []

    dt = datetime.fromisoformat(tx['timestamp'])
    hour = dt.hour
    if hour < 6:
        score += 2
        rules.append('R3')

    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')

    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')
    
    return score, rules

for message in consumer:
    tx = message.value
    score, rules = score_transaction(tx)
    if score >= 3:
        tx['score'] = score
        tx['rules'] = rules
        tx['suspicious'] = True
        alert_producer.send('alerts', value=tx)
        print(f'ALERT: {tx}')

alert_producer.flush()

Overwriting scoring_consumer.py


In [39]:
# Uruchomienie jednoczesnie w terminalu:
# > python3 consumer_filter.py & python3 scoring_consumer.py & python3 producer.py